In [2]:
# # lib installations
# !pip install -U \
#   llama-index-core \
#   llama-index-embeddings-huggingface \
#   llama-index-vector-stores-faiss \
#   llama-index-llms-ollama \
#   faiss-cpu

In [2]:
from pathlib import Path

BASE_DIR = Path(".").resolve()       # this will be your storied/ folder
DATA_DIR = BASE_DIR / "data"         # or "sdp", wherever your .sdp files live
INDEX_DIR = BASE_DIR / "index"
INDEX_DIR.mkdir(exist_ok=True)


# Approach 3: Local RAG Pipeline with LlamaIndex, FAISS, and Evaluation

This notebook is a starting point for building a fully local, open‑source RAG (Retrieval‑Augmented Generation) system that:

- Ingests and indexes Storied Data reports (e.g., `.sdp` / JSON / text reports).
- Uses local embeddings and a FAISS-based vector store for semantic retrieval.
- Uses a local LLM (e.g., via Ollama) to synthesize a combined report from multiple source documents.
- Provides hooks for RAG evaluation with tools like RAGAS and TruLens.

The goal is not to generate new SDP files, but to consume existing generated reports, retrieve their content, and combine them into a new, unified analytical report.



## Table of Contents

1. [Project Setup and Configuration](#1-Project-Setup-and-Configuration)  
2. [Data Ingestion: Storied Data Reports](#2-Data-Ingestion-Storied-Data-Reports)  
3. [Embedding and Vector Store (FAISS)](#3-Embedding-and-Vector-Store-FAISS)  
4. [LLM Integration (Local LLM via Ollama or Equivalent)](#4-LLM-Integration-Local-LLM-via-Ollama-or-Equivalent)  
5. [Building the RAG Query Engine](#5-Building-the-RAG-Query-Engine)  
6. [Combined Report Generation Workflow](#6-Combined-Report-Generation-Workflow)  
7. [RAG Evaluation Hooks (RAGAS, TruLens)](#7-RAG-Evaluation-Hooks-RAGAS-TruLens)  
8. [Next Steps and Extension Ideas](#8-Next-Steps-and-Extension-Ideas)



---
## 1. Project Setup and Configuration

This section defines the core configuration and environment dependencies.

We assume:
- You have a directory of Storied Data reports (e.g., `.sdp` JSON, JSON exports, or text reports).
- You want to keep everything local and open‑source, including embeddings and LLM inference.
- You have (or plan to have) a local LLM runtime such as Ollama running on your machine.


In [3]:
# 1.1 Optional: install dependencies (run once per environment)
# NOTE: These are example commands; uncomment and adapt as needed.
# !pip install llama-index-core llama-index-llms-ollama llama-index-vector-stores-faiss
# !pip install sentence-transformers faiss-cpu chromadb
# !pip install ragas trulens-eval

import os
from pathlib import Path

# Base directories (adapt these paths to your environment)
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"          # Directory containing Storied Data reports
INDEX_DIR = PROJECT_ROOT / "index"        # Directory where FAISS index / artifacts will be stored
INDEX_DIR.mkdir(exist_ok=True, parents=True)

# Embedding and LLM configuration
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # or 'BAAI/bge-small-en-v1.5'
USE_FAISS = True  # if False, LlamaIndex can fall back to in-memory index

# LLM (local) configuration example: Ollama
LOCAL_LLM_MODEL = "llama3:8b"  # example for Ollama; change as needed
OLLAMA_ENDPOINT = "http://localhost:11434"  # default Ollama endpoint

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Index directory: {INDEX_DIR}")


Project root: /Users/atheeshkrishnan/AK/DEV/hawkdove/storied
Data directory: /Users/atheeshkrishnan/AK/DEV/hawkdove/storied/data
Index directory: /Users/atheeshkrishnan/AK/DEV/hawkdove/storied/index


In [4]:
import os

os.makedirs("data", exist_ok=True)
print("data folder created:", os.listdir())

data folder created: ['approach3_llamaindex_faiss_rag_start.ipynb', 'index', 'venv', 'data']



---
## 2. Data Ingestion: Storied Data Reports

In this project, Storied Data reports are the primary data source.  
They may be stored as:

- `.sdp` files (JSON-like structure with aggregated commentary fields), or  
- `.json` files, or  
- `.txt` / `.md` textual exports.

The ingestion goal is to:
1. Load all relevant reports from a directory.
2. Extract the commentary / analytical content (e.g., the `"Comments"` field).
3. Wrap the content into LlamaIndex `Document` objects with appropriate metadata (e.g., source name, date, report type).


In [5]:
# 2.1 Example: helper to load and normalize Storied Data reports
import json
from typing import List, Dict, Any

try:
    from llama_index.core import Document
except ImportError:
    Document = None  # placeholder if llama-index is not installed yet


def load_sdp_file(path: Path) -> List[Dict[str, Any]]:
    """
    Load a single .sdp JSON file and extract commentary.
    This is a generic template; adapt the extraction logic to your actual Storied Data schema.

    Expected pattern (example):
    - Top-level JSON with a 'columns' array.
    - One of the columns has title 'Comments' and a 'values' list.

    Returns a list of dicts, each with:
    - 'text': comment text
    - 'metadata': {'source': filename, ...}
    """
    data = json.loads(path.read_text(encoding="utf-8", errors="ignore"))
    comments = []

    columns = data.get("columns", [])
    for col in columns:
        if (col.get("title") or "").strip().lower() == "comments" and isinstance(col.get("values"), list):
            for idx, val in enumerate(col["values"]):
                if not isinstance(val, str) or not val.strip():
                    continue
                comments.append({
                    "text": val.strip(),
                    "metadata": {
                        "source": path.name,
                        "comment_index": idx,
                        # Add more metadata if available, e.g., date, category, etc.
                    },
                })
            break
    return comments


def load_all_reports(data_dir: Path) -> List[Dict[str, Any]]:
    """
    Load all Storied Data reports from the given directory.
    This function can be extended to handle multiple formats (.sdp, .json, .txt, etc.).
    """
    all_entries: List[Dict[str, Any]] = []

    for path in sorted(data_dir.glob("**/*")):
        if path.suffix.lower() == ".sdp":
            entries = load_sdp_file(path)
        elif path.suffix.lower() in {".json"}:
            # Optionally handle generic JSON reports here
            try:
                data = json.loads(path.read_text(encoding="utf-8", errors="ignore"))
                # TODO: define how to extract commentary from generic JSON
                entries = []
            except Exception:
                entries = []
        elif path.suffix.lower() in {".txt", ".md"}:
            text = path.read_text(encoding="utf-8", errors="ignore")
            entries = [{"text": text, "metadata": {"source": path.name}}]
        else:
            continue

        all_entries.extend(entries)

    print(f"Loaded {len(all_entries)} total commentary entries from {data_dir}.")
    return all_entries


# 2.2 Convert raw entries into LlamaIndex Documents
def build_documents(entries: List[Dict[str, Any]]) -> List["Document"]:
    if Document is None:
        raise ImportError("llama-index-core is not installed. Please install it to use Document objects.")
    docs: List[Document] = []
    for e in entries:
        docs.append(Document(text=e["text"], metadata=e.get("metadata", {})))
    return docs


# Example: only run after DATA_DIR is populated
if DATA_DIR.exists():
    raw_entries = load_all_reports(DATA_DIR)
    print(f"Example entry: {raw_entries[0] if raw_entries else 'N/A'}")
else:
    print("DATA_DIR does not exist yet; please create it and add Storied Data reports.")


Loaded 1417 total commentary entries from /Users/atheeshkrishnan/AK/DEV/hawkdove/storied/data.
Example entry: {'text': '2021 will get worse before it gets better', 'metadata': {'source': 'TSI Global Index.sdp', 'comment_index': 0}}


In [6]:
import sys
print(sys.executable)

/Users/atheeshkrishnan/AK/DEV/hawkdove/storied/venv/bin/python



---
## 3. Embedding and Vector Store (FAISS)

This section builds the vector index used for semantic retrieval.

Core ideas:
- Use a local embedding model (e.g., SentenceTransformers) to embed each document chunk.
- Use FAISS as the underlying vector store for efficient similarity search.
- Wrap FAISS with LlamaIndex's vector store abstraction, so we can use LlamaIndex's query engines.


In [13]:
# ======================================================
# 3. Build LlamaIndex vector index (in-memory only)
# ======================================================

from typing import Optional

try:
    from llama_index.core import VectorStoreIndex, Settings
    from llama_index.core.node_parser import SentenceSplitter
    from llama_index.embeddings.huggingface import HuggingFaceEmbedding
except ImportError as e:
    VectorStoreIndex = None
    Settings = None
    SentenceSplitter = None
    HuggingFaceEmbedding = None
    print("LlamaIndex core / HF embeddings not installed:", e)


def build_index_from_documents(documents) -> Optional["VectorStoreIndex"]:
    """
    Build a LlamaIndex VectorStoreIndex in memory.

    Steps:
    - Configure a HuggingFace embedding model (same model name as EMBEDDING_MODEL_NAME).
    - Register it in LlamaIndex global Settings.
    - Split documents into smaller nodes (chunks) with SentenceSplitter.
    - Build an in-memory VectorStoreIndex over those nodes.

    Returns:
        VectorStoreIndex or None if something fails.
    """
    if VectorStoreIndex is None or Settings is None:
        raise ImportError(
            "llama-index-core and llama-index-embeddings-huggingface must be installed."
        )

    # 1) Set up embedding model
    embed_model = HuggingFaceEmbedding(model_name=EMBEDDING_MODEL_NAME)
    Settings.embed_model = embed_model

    # (Optional) sanity check
    print("Embedding model:", embed_model)
    try:
        dummy_vec = embed_model.get_text_embedding("test")
        print("Embedding dim:", len(dummy_vec))
    except Exception as e:
        print("Embedding sanity check failed:", e)

    # 2) Chunk documents into nodes
    splitter = SentenceSplitter(chunk_size=1024, chunk_overlap=128)
    nodes = splitter.get_nodes_from_documents(documents)
    print(f"Generated {len(nodes)} nodes from {len(documents)} documents.")

    # 3) Build in-memory index
    index = VectorStoreIndex(nodes)
    print("In-memory LlamaIndex VectorStoreIndex built successfully.")
    return index


# ------------------------------------------------------
# Build the index once, after raw_entries/documents exist
# ------------------------------------------------------
INDEX: Optional["VectorStoreIndex"] = None

if DATA_DIR.exists() and "raw_entries" in globals() and raw_entries:
    try:
        documents = build_documents(raw_entries)
        INDEX = build_index_from_documents(documents)
    except Exception as e:
        print("Index build skipped due to error:", e)
else:
    print("Index not built yet. Ensure DATA_DIR has data and raw_entries are loaded.")


Embedding model: model_name='sentence-transformers/all-MiniLM-L6-v2' embed_batch_size=10 callback_manager=<llama_index.core.callbacks.base.CallbackManager object at 0x16a2d4790> num_workers=None embeddings_cache=None max_length=256 normalize=True query_instruction=None text_instruction=None cache_folder=None show_progress_bar=False
Embedding dim: 384
Generated 1417 nodes from 1417 documents.
In-memory LlamaIndex VectorStoreIndex built successfully.


In [14]:
print("Embedding model:", Settings.embed_model)
print("Embedding dim:", Settings.embed_model._model.get_sentence_embedding_dimension())

Embedding model: model_name='sentence-transformers/all-MiniLM-L6-v2' embed_batch_size=10 callback_manager=<llama_index.core.callbacks.base.CallbackManager object at 0x16a2d4790> num_workers=None embeddings_cache=None max_length=256 normalize=True query_instruction=None text_instruction=None cache_folder=None show_progress_bar=False
Embedding dim: 384


In [15]:
query_engine = INDEX.as_query_engine()
response = query_engine.query("What topics are covered in the dataset?")
print(response)

ConnectionError: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download


---
## 4. LLM Integration (Local LLM via Ollama or Equivalent)

This section configures the local LLM used for generation.

We assume:
- You have a local LLM runtime such as Ollama running (e.g., `ollama run llama3:8b`).
- We use the LlamaIndex LLM wrapper to abstract the model backend.

You can swap the local backend later (e.g., vLLM, text-generation-webui) as long as it exposes an HTTP or Python API.


In [9]:
# ======================================================
# 4. Configure LLM (Ollama) and build a query helper
# ======================================================

TOP_K = 5  # number of similar chunks to retrieve per question

from typing import Optional

try:
    from llama_index.llms.ollama import Ollama
    from llama_index.core import Settings
except ImportError as e:
    Ollama = None
    Settings = None
    print("LlamaIndex Ollama / core modules not fully installed:", e)


def init_ollama_llm(model_name: str = "llama3:8b") -> Optional["Ollama"]:
    """
    Try to initialize a local LLM via Ollama.

    Requirements (when you run this on your own machine, not Colab):
      - Ollama installed: https://ollama.com/
      - ollama serve running
      - Model pulled, e.g.: `ollama pull llama3:8b`

    In Colab this will normally fail (no local Ollama daemon), but the error
    message will be explicit so you know what to do when running locally.
    """
    if Ollama is None or Settings is None:
        print("Ollama or LlamaIndex core not available; cannot configure local LLM.")
        return None

    try:
        llm = Ollama(
            model=model_name,
            request_timeout=120.0,  # generous timeout for local inference
        )
        # Register in global Settings so LlamaIndex knows which LLM to use
        Settings.llm = llm
        print(f"Ollama LLM configured with model: {model_name}")
        return llm
    except Exception as e:
        print(
            "Could not configure Ollama LLM.\n"
            "Make sure Ollama is installed and running locally, "
            "and that the model is pulled (e.g. `ollama pull llama3:8b`)."
        )
        print("Underlying error:", e)
        return None


LLM = init_ollama_llm(model_name="llama3:8b")


def answer(question: str, top_k: int = TOP_K) -> str:
    """
    High-level helper:
      - Uses the global INDEX built in Section 3.
      - Retrieves top_k most relevant nodes.
      - Asks the configured LLM (if available) to answer based on those nodes.

    If no LLM is configured, it will fall back to returning the top snippets
    so you still get *some* output.
    """
    if INDEX is None:
        raise RuntimeError("INDEX is not built yet. Run the indexing section first.")

    # Build a query engine; if LLM is registered in Settings.llm it will be used automatically.
    # We can also pass `llm=LLM` explicitly to be safe.
    query_engine = INDEX.as_query_engine(
        similarity_top_k=top_k,
        response_mode="compact",
        llm=LLM if LLM is not None else None,
    )

    if LLM is None:
        print("WARNING: No LLM configured; returning a synthesized text-only response from retrieved nodes.")
    response = query_engine.query(question)
    return str(response)


Ollama LLM configured with model: llama3:8b


In [10]:
# 4.1 Configure LLM in LlamaIndex
try:
    from llama_index.llms.ollama import Ollama
    from llama_index.core import ServiceContext
except ImportError:
    Ollama = None
    ServiceContext = None


def build_service_context() -> Optional["ServiceContext"]:
    if Ollama is None or ServiceContext is None:
        raise ImportError("llama-index-llms-ollama is not installed. Please install it to use a local Ollama LLM.")
    llm = Ollama(model=LOCAL_LLM_MODEL, base_url=OLLAMA_ENDPOINT)
    service_context = ServiceContext.from_defaults(llm=llm, embed_model=Settings.embed_model)
    return service_context


SERVICE_CONTEXT: Optional["ServiceContext"] = None
if INDEX is not None:
    try:
        SERVICE_CONTEXT = build_service_context()
        print("Service context with local LLM initialized.")
    except Exception as e:
        print("Service context not initialized:", e)
else:
    print("Service context not created because INDEX is None.")

Service context not initialized: ServiceContext is deprecated. Use llama_index.settings.Settings instead, or pass in modules to local functions/methods/interfaces.
See the docs for updated usage/migration: 
https://docs.llamaindex.ai/en/stable/module_guides/supporting_modules/service_context_migration/



---
## 5. Building the RAG Query Engine

With the index and LLM configured, we now build a RAG query engine that:

1. Takes a natural language query.
2. Retrieves the top‑K most relevant nodes from the vector index.
3. Feeds them, along with the query, into the local LLM.
4. Produces a synthesized response, optionally with references to source documents.


In [ ]:
# 5.1 Construct LlamaIndex query engine
from typing import Any

QUERY_ENGINE: Any = None

if INDEX is not None and SERVICE_CONTEXT is not None:
    QUERY_ENGINE = INDEX.as_query_engine(
        similarity_top_k=10,
        service_context=SERVICE_CONTEXT,
        response_mode="tree_summarize",  # can be changed to 'compact' or 'refine'
    )
    print("Query engine initialized.")
else:
    print("Query engine not initialized. Ensure INDEX and SERVICE_CONTEXT are available.")


# 5.2 Simple helper to query the engine
def ask(query: str) -> Any:
    if QUERY_ENGINE is None:
        raise RuntimeError("QUERY_ENGINE is not initialized.")
    response = QUERY_ENGINE.query(query)
    return response


# Example (when everything is configured):
# resp = ask("Summarize the key macroeconomic themes across all reports.")
# print(resp)



---
## 6. Combined Report Generation Workflow

The core use case is to take multiple Storied Data reports and generate a new combined report that:

- Synthesizes themes across all documents.
- Highlights key macroeconomic narratives (inflation, labor market, growth, policy stance, risks).
- Optionally uses a fixed structure (sections, bullet points) suitable for downstream consumption.


In [ ]:
# 6.1 Example: generate a combined report

COMBINED_REPORT_PROMPT = """
You are a macroeconomic research analyst.

You are given retrieved excerpts from multiple Storied Data reports
that discuss financial markets, macro conditions, and policy themes.

Task:
- Produce a single, unified report that synthesizes the main themes.
- Organize the report into sections:
  1) Executive Summary
  2) Inflation
  3) Labor Market
  4) Growth and Activity
  5) Financial Conditions and Markets
  6) Policy Outlook and Risks

Guidelines:
- Be concise but substantive (2–5 bullet points per section).
- Use only information grounded in the retrieved context.
- Avoid speculation or external knowledge.
- Do not mention that you are an AI model.
"""


def generate_combined_report() -> Any:
    if QUERY_ENGINE is None:
        raise RuntimeError("QUERY_ENGINE is not initialized.")
    query = (
        COMBINED_REPORT_PROMPT
        + "\n\nQuestion: Generate the combined macroeconomic report now, based on all available context."
    )
    response = QUERY_ENGINE.query(query)
    return response


# Example:
# combined_resp = generate_combined_report()
# print(combined_resp)



---
## 7. RAG Evaluation Hooks (RAGAS, TruLens)

To move beyond “it feels right”, we can introduce evaluation tooling:

- RAGAS: Framework for evaluating RAG pipelines on metrics like:
  - Faithfulness (is the answer grounded in context?)
  - Context relevance
  - Answer semantic similarity to ground truth
- TruLens: Observability and evaluation toolkit for LLM systems, including:
  - Tracing which context chunks were used
  - Measuring hallucination risk
  - Inspecting prompt/response pairs

In this notebook, we provide scaffolding to integrate these tools later.


In [ ]:
# 7.1 RAGAS scaffolding (to be filled in when ground-truth data is available)

# NOTE: This is a template for future work.
# You will need a dataset of (question, ground_truth_answer, retrieved_context) pairs
# to properly run RAGAS metrics.

# from datasets import Dataset
# from ragas import evaluate
# from ragas.metrics import faithfulness, answer_relevancy, context_relevancy

# # Example placeholder structure:
# eval_data = {
#     "question": [
#         "What are the main inflation themes across reports?",
#     ],
#     "answer": [
#         "<MODEL_GENERATED_ANSWER_HERE>",
#     ],
#     "contexts": [
#         ["<RETRIEVED_CONTEXT_CHUNK_1>", "<RETRIEVED_CONTEXT_CHUNK_2>"]
#     ],
#     "ground_truth": [
#         "<CURATED_GROUND_TRUTH_SUMMARY_HERE>",
#     ],
# }

# dataset = Dataset.from_dict(eval_data)
# result = evaluate(
#     dataset,
#     metrics=[faithfulness, answer_relevancy, context_relevancy],
# )
# print(result)


# 7.2 TruLens scaffolding (for tracing and observability)

# from trulens_eval import Tru
# from trulens_eval.tru_llamaindex import TruLlamaIndexApp

# tru = Tru()

# if INDEX is not None and QUERY_ENGINE is not None:
#     app = TruLlamaIndexApp(
#         query_engine=QUERY_ENGINE,
#         app_id="storieddata_rag_v1",
#     )
#     tru.run_dashboard()  # Optional: start dashboard to inspect traces



---
## 8. Next Steps and Extension Ideas

This notebook is a starting point. Possible next steps:

1. Improve ingestion  
   - Fine-tune the `.sdp` parsing logic (dates, categories, risk flags).
   - Add support for more formats (PDF → text, HTML pages, etc.).

2. Refine retrieval  
   - Experiment with different embedding models (e.g., `bge-small`, `gte-large`).  
   - Tune `chunk_size`, `chunk_overlap`, and `similarity_top_k`.

3. Enhance report generation  
   - Enforce stricter structure or JSON output for downstream systems.  
   - Add multiple “modes”: high-level summary, risk-only report, theme comparison across time.

4. Introduce proper evaluation  
   - Curate a small labeled set of Q&A pairs from your reports.  
   - Use RAGAS to get repeatable metrics.  
   - Use TruLens to inspect which chunks drive the model’s reasoning.

5. Modularize into a package  
   - Extract ingestion, indexing, query, and evaluation into separate modules.  
   - Add CLI or simple web UI (e.g., Streamlit, Gradio) for non-technical users.

From here, you can edit each section case by case to align with your exact data schema, local LLM setup, and evaluation needs.
